In [2]:
import numpy as np
import pandas as pd

sentinels = [99999]
ratio_round_val = 4 # Decimal rounding



# -------------------------------------------------------------------------
# LOAD/ANALYZE DATASET
# -------------------------------------------------------------------------
df = pd.read_csv('../data/insurance_api.csv')
df.columns = df.columns.str.upper()

# Prints out columns by sentinel count
sentinels_per_col = df.isin(sentinels).sum()
print("\nTop columns ranked by sentinel count:")
print(sentinels_per_col[sentinels_per_col > 0].sort_values(ascending=False))
print(
    f"Starting shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns."
)


# -------------------------------------------------------------------------
# FILTER OUT TERMINATED POLICIES
# -------------------------------------------------------------------------
end_year_cols = [
    "PL_END_YEAR",
    "CL_END_YEAR",
    "COMMISIONS_END_YEAR",
    "ACTIVITY_NOTES_END_YEAR",
]

# Identifies all rows where any end year is valid (not null or sentinel)
terminated_mask = pd.Series(False, index=df.index)
for col in end_year_cols:
    if col in df.columns:
        valid_end_year = (
            df[col].notna() & ~df[col].astype(str).str.contains("99999")
        )
        terminated_mask = terminated_mask | valid_end_year

# Drops all rows in terminated_mask (only tracking the future of currently running policies)
df = df[~terminated_mask].copy()
print(f"Terminated records removed: {terminated_mask.sum():,} ({(terminated_mask.sum()/len(df))*100:.2f}%)")


# -------------------------------------------------------------------------
# SENTINEL CLEANUP
# -------------------------------------------------------------------------
# Payment methods
platform_cols = [c for c in df.columns if "_QUO_CT_" in c or "_BOUND_CT_" in c]
for col in platform_cols:
    df[col] = df[col].replace(sentinels, 0)

# Calculable ratios
ratio_cols = [
    "RETENTION_RATIO",
    "GROWTH_RATE_3YR",
    "LOSS_RATIO",
    "LOSS_RATIO_3YR",
]
for col in ratio_cols:
    if col in df.columns:
        df[col] = df[col].replace(sentinels, np.nan)

# Primary Agency ID
df["PRIMARY_AGENCY_ID"] = np.where(
    df["PRIMARY_AGENCY_ID"].astype(str).str.contains("99999"),
    df["AGENCY_ID"],
    df["PRIMARY_AGENCY_ID"],
)

# Premiums
prior_cols = [
    "PREV_POLY_INFORCE_QTY",
    "PREV_WRTN_PREM_AMT",
    "NB_WRTN_PREM_AMT",
]
for col in prior_cols:
    if col in df.columns:
        df[col] = df[col].replace(sentinels, 0)

# Other numerical values
demo_cols = [
    "ACTIVE_PRODUCERS",
    "MIN_AGE",
    "MAX_AGE",
    "AGENCY_APPOINTMENT_YEAR",
]
for col in demo_cols:
    if col in df.columns:
        df[col] = df[col].replace(sentinels, np.nan)



# -------------------------------------------------------------------------
# CONSOLIDATE PAYMENT COLUMNS
# -------------------------------------------------------------------------
pl_quo_cols = [c for c in df.columns if c.startswith("PL_QUO_CT_")]
pl_bound_cols = [c for c in df.columns if c.startswith("PL_BOUND_CT_")]
cl_quo_cols = [c for c in df.columns if c.startswith("CL_QUO_CT_")]
cl_bound_cols = [c for c in df.columns if c.startswith("CL_BOUND_CT_")]

all_platform_subcols = (
    pl_quo_cols + pl_bound_cols + cl_quo_cols + cl_bound_cols
)
# Removes sentinels and turns all values into floats
df[all_platform_subcols] = df[all_platform_subcols].fillna(0).astype(float)

# Adds columns (axis=1) across for each payment category, and creates new columns
df["PL_QUO_TOTAL"] = df[pl_quo_cols].sum(axis=1)
df["PL_BOUND_TOTAL"] = df[pl_bound_cols].sum(axis=1)
df["CL_QUO_TOTAL"] = df[cl_quo_cols].sum(axis=1)
df["CL_BOUND_TOTAL"] = df[cl_bound_cols].sum(axis=1)


# Consolidates QUO and BOUND CTs (as PROD_LINE already gives us prerequisite info)
df["QUO_CT"] = np.where(
    df["PROD_LINE"] == "CL", df["CL_QUO_TOTAL"], df["PL_QUO_TOTAL"]
)
df["BOUND_CT"] = np.where(
    df["PROD_LINE"] == "CL", df["CL_BOUND_TOTAL"], df["PL_BOUND_TOTAL"]
)

# Calculates HIT_RATIO (bound / quote)
df["HIT_RATIO"] = np.where(
    df["QUO_CT"] > 0, df["BOUND_CT"] / df["QUO_CT"], 0.0
)



# -------------------------------------------------------------------------
# LEGACY ROW FILTERING
# -------------------------------------------------------------------------
activity_metrics = [
    "QUO_CT",
    "BOUND_CT",
    "POLY_INFORCE_QTY",
    "RETENTION_POLY_QTY",
    "WRTN_PREM_AMT",
    "NB_WRTN_PREM_AMT",
    "PRD_ERND_PREM_AMT",
]
# Replaces empty cells with 0s
df[activity_metrics] = df[activity_metrics].fillna(0)

# Flags rows where all relevant columns are 0 (legacy records not removed from dataset yet)
ghost_mask = (df[activity_metrics] == 0).all(axis=1)

# Prints legacy percentage to 2 decimal places, then drops rows
print(f"Legacy records removed: {ghost_mask.sum():,} ({(ghost_mask.sum()/len(df))*100:.2f}%)")
df = df[~ghost_mask].copy()



# -------------------------------------------------------------------------
# DEMOGRAPHIC AND METADATA TRANSFORMATIONS
# -------------------------------------------------------------------------
df["STAT_PROFILE_DATE_YEAR"] = pd.to_numeric(
    df["STAT_PROFILE_DATE_YEAR"], errors="coerce"
)
df["AGENCY_APPOINTMENT_YEAR"] = pd.to_numeric(
    df["AGENCY_APPOINTMENT_YEAR"], errors="coerce"
)

# Creates tenure years and bounds lower threshold of 0
df["TENURE_YEARS"] = (
    df["STAT_PROFILE_DATE_YEAR"] - df["AGENCY_APPOINTMENT_YEAR"]
).clip(lower=0)

# Fills in NaNs with state-wide medians (transform requires a function input, resulting in lambda inline function)
df["TENURE_YEARS"] = df.groupby("STATE_ABBR")["TENURE_YEARS"].transform(
    lambda x: x.fillna(x.median())
)

# Fills in any remaining NaNs with the national median (fallback)
df["TENURE_YEARS"] = df["TENURE_YEARS"].fillna(df["TENURE_YEARS"].median())

# Ages
df["MIN_AGE"] = pd.to_numeric(df["MIN_AGE"], errors="coerce")
df["MAX_AGE"] = pd.to_numeric(df["MAX_AGE"], errors="coerce")
df["AVG_AGE"] = (df["MIN_AGE"] + df["MAX_AGE"]) / 2
overall_median_age = df["AVG_AGE"].median()
df["AVG_AGE"] = df["AVG_AGE"].fillna(
    overall_median_age if pd.notna(overall_median_age) else 45.0
)

# Active Producers (sets lower bound of 1)
df["ACTIVE_PRODUCERS"] = (
    pd.to_numeric(df["ACTIVE_PRODUCERS"], errors="coerce")
    .fillna(1)
    .replace(0, 1)
)



# -------------------------------------------------------------------------
# ANNUALIZATION BASED ON MONTHS
# -------------------------------------------------------------------------
df["MONTHS"] = pd.to_numeric(df["MONTHS"], errors="coerce")

# Sets all invalid rows to 12
df["MONTHS"] = np.where(
    df["MONTHS"].isna() | (df["MONTHS"] <= 0) | (df["MONTHS"] > 12),
    12.0,
    df["MONTHS"],
)
df["ANNUALIZATION_FACTOR"] = 12.0 / df["MONTHS"]

annualize_cols = [
    "WRTN_PREM_AMT",
    "NB_WRTN_PREM_AMT",
    "PRD_ERND_PREM_AMT",
    "PRD_INCRD_LOSSES_AMT",
    "PREV_WRTN_PREM_AMT",
    "QUO_CT",
    "BOUND_CT",
]

# For every financial column, annualizes data based on months and rounds to 2 decimal places
for col in annualize_cols:
    if col in df.columns:
        df[f"{col}_ANNUAL"] = (
            df[col].fillna(0) * df["ANNUALIZATION_FACTOR"]
        ).round(2)

# Calculates the premium per producer
df["PREM_PER_PRODUCER"] = (
    df["WRTN_PREM_AMT_ANNUAL"].clip(lower=0) / df["ACTIVE_PRODUCERS"]
).round(2)




# -------------------------------------------------------------------------
# CONSOLIDATE RATIOS & FIX GROWTH RATE BUG
# -------------------------------------------------------------------------
# Calculates Retention Ratio (accounts for division by 0)
if "PREV_POLY_INFORCE_QTY" in df.columns and "RETENTION_POLY_QTY" in df.columns:
    df["RETENTION_RATIO"] = np.where(
        df["PREV_POLY_INFORCE_QTY"] > 0,
        df["RETENTION_POLY_QTY"] / df["PREV_POLY_INFORCE_QTY"],
        np.nan,
    )
    # Sets lower and upper bounds (0, 1)
    df["RETENTION_RATIO"] = df["RETENTION_RATIO"].clip(lower=0.0, upper=1.0)

# Flag new agencies
df["IS_NEW_AGENCY"] = np.where(df["TENURE_YEARS"] < 3, 1, 0)
df.loc[df["TENURE_YEARS"] < 3, ["LOSS_RATIO_3YR", "GROWTH_RATE_3YR"]] = np.nan

# Unified Loss Ratio (uses 3-year when possible, otherwise defaults to 1-year)
df["UNIFIED_LOSS_RATIO"] = np.where(
    df["TENURE_YEARS"] >= 3,
    df["LOSS_RATIO_3YR"].fillna(df["LOSS_RATIO"]),
    df["LOSS_RATIO"],
)

df["UNIFIED_LOSS_RATIO"] = df["UNIFIED_LOSS_RATIO"].clip(lower=-1.0, upper=5.0)

# Unified Growth Rate (Fixes division by 0 and bounds extreme outliers)
if "PREV_WRTN_PREM_AMT_ANNUAL" in df.columns:
    valid_baseline = df["PREV_WRTN_PREM_AMT_ANNUAL"] > 100
    df["GROWTH_RATE_1YR"] = np.where(
        valid_baseline,
        (df["WRTN_PREM_AMT_ANNUAL"] - df["PREV_WRTN_PREM_AMT_ANNUAL"])
        / df["PREV_WRTN_PREM_AMT_ANNUAL"],
        np.nan,
    )
else:
    df["GROWTH_RATE_1YR"] = np.nan

df["UNIFIED_GROWTH_RATE"] = np.where(
    df["TENURE_YEARS"] >= 3,
    df["GROWTH_RATE_3YR"].fillna(df["GROWTH_RATE_1YR"]),
    df["GROWTH_RATE_1YR"],
)

df["UNIFIED_GROWTH_RATE"] = (
    df["UNIFIED_GROWTH_RATE"].fillna(0.0).clip(lower=-1.0, upper=3.0)
).round(ratio_round_val)

# Fixes missing values with LOSS_RATIO and RETENTION_RATIO medians by PROD_LINE (lambda)
df["UNIFIED_LOSS_RATIO"] = df.groupby("PROD_LINE")[
    "UNIFIED_LOSS_RATIO"
].transform(lambda x: x.fillna(x.median()))
df["UNIFIED_LOSS_RATIO"] = df["UNIFIED_LOSS_RATIO"].fillna(
    df["UNIFIED_LOSS_RATIO"].median()
).round(ratio_round_val)

df["RETENTION_RATIO"] = df.groupby("STATE_ABBR")["RETENTION_RATIO"].transform(
    lambda x: x.fillna(x.median())
)
df["RETENTION_RATIO"] = df["RETENTION_RATIO"].fillna(
    df["RETENTION_RATIO"].median()
).round(ratio_round_val)



# -------------------------------------------------------------------------
# FINAL DATASET FORMATTING
# -------------------------------------------------------------------------
logical_column_order = [
    # Identifiers & Categorization
    "AGENCY_ID",
    "PRIMARY_AGENCY_ID",
    "STATE_ABBR",
    "STAT_PROFILE_DATE_YEAR",
    "PROD_LINE",
    "PROD_ABBR",
    # Agency Profile & Demographics
    "TENURE_YEARS",
    "IS_NEW_AGENCY",
    "ACTIVE_PRODUCERS",
    "AVG_AGE",
    # Funnel & Policy Activity
    "QUO_CT_ANNUAL",
    "BOUND_CT_ANNUAL",
    "HIT_RATIO",
    "POLY_INFORCE_QTY",
    "PREV_POLY_INFORCE_QTY",
    "RETENTION_RATIO",
    # Financial Dollar Volumes
    "WRTN_PREM_AMT_ANNUAL",
    "PREV_WRTN_PREM_AMT_ANNUAL",
    "NB_WRTN_PREM_AMT_ANNUAL",
    "PRD_ERND_PREM_AMT_ANNUAL",
    "PRD_INCRD_LOSSES_AMT_ANNUAL",
    # Bottom-Line KPIs
    "PREM_PER_PRODUCER",
    "UNIFIED_GROWTH_RATE",
    "UNIFIED_LOSS_RATIO",
]

# Drops all unnecessary rows in the process
df = df[logical_column_order]

# Renames rows for Excel simplicity
df = df.rename(columns={
    "STATE_ABBR": "STATE",
    "STAT_PROFILE_DATE_YEAR": "YEAR",
    "WRTN_PREM_AMT_ANNUAL": "WRTN_PREM",
    "PREV_WRTN_PREM_AMT_ANNUAL": "PREV_WRTN_PREM",
    "NB_WRTN_PREM_AMT_ANNUAL": "NEW_BUSINESS_WRTN_PREM",
    "PRD_ERND_PREM_AMT_ANNUAL": "PRD_ERND_PREM",
    "PRD_INCRD_LOSSES_AMT_ANNUAL": "PRD_INCRD_LOSSES",
    "UNIFIED_GROWTH_RATIO": "GROWTH_RATE",
    "UNIFIED_LOSS_RATIO": "LOSS_RATIO"
})

df.columns = df.columns.str.replace('_', ' ').str.title()

df.to_csv('../data/insurance_data_clean.csv', index=False)
print(
    f"Cleaned dataset exported successfully! Final shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns."
)


Top columns ranked by sentinel count:
ACTIVITY_NOTES_END_YEAR      209258
PL_END_YEAR                  209174
COMMISIONS_END_YEAR          205604
CL_END_YEAR                  204760
ACTIVITY_NOTES_START_YEAR    171536
COMMISIONS_START_YEAR        141937
RETENTION_RATIO              129978
CL_START_YEAR                129311
CL_QUO_CT_SBZ                 98689
CL_BOUND_CT_SBZ               98689
CL_QUO_CT_MDS                 98689
CL_BOUND_CT_MDS               98689
CL_BOUND_CT_EQT               98689
CL_QUO_CT_EQT                 98689
GROWTH_RATE_3YR               91067
PL_START_YEAR                 82200
PL_QUO_CT_ELINKS              57444
PL_BOUND_CT_PLRANK            57444
PL_QUO_CT_PLRANK              57444
PL_BOUND_CT_EQTTE             57444
PL_BOUND_CT_ELINKS            57444
PL_QUO_CT_EQTTE               57444
PL_BOUND_CT_APPLIED           57444
PL_QUO_CT_APPLIED             57444
PL_BOUND_CT_TRANSACTNOW       57444
PL_QUO_CT_TRANSACTNOW         57444
LOSS_RATIO               